# 03 — Optimisation Multi-Objectifs des Hyperparamètres

**Projet : Optimisation multi-objectifs pour AutoML — Option 2**

Ce notebook compare **4 algorithmes d'optimisation** appliqués aux hyperparamètres de 3 modèles ML classiques.

## Algorithmes comparés
| # | Méthode | Bibliothèque | Type |
|---|---------|-------------|------|
| 1 | **NSGA-II** | pymoo | Évolutionnaire multi-objectifs |
| 2 | **Bayesian Optimization** | Optuna (TPESampler) | Probabiliste bayésien |
| 3 | **Hyperband** | Optuna (HyperbandSampler) | Successive Halving adaptatif |
| 4 | **Genetic Algorithm** | pymoo (GA) | Évolutionnaire mono-objectif |

## Objectifs d'optimisation
- **Objectif 1 :** Maximiser l'accuracy de validation
- **Objectif 2 :** Minimiser le temps d'inférence (ms/image)

## Modèle principal : Random Forest
Choisi pour sa robustesse, sa rapidité d'entraînement et l'intérêt de ses hyperparamètres.

## Espace de recherche
```
n_estimators   : [10, 50, 100, 200, 300]
max_depth      : [5, 10, 15, 20, None]
min_samples_split : [2, 5, 10]
max_features   : ['sqrt', 'log2', 0.3, 0.5]
```

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Librairies de base chargées.")

# Test pymoo
try:
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.algorithms.soo.nonconvex.ga import GA
    from pymoo.core.problem import Problem
    from pymoo.optimize import minimize
    from pymoo.operators.crossover.sbx import SBX
    from pymoo.operators.mutation.pm import PM
    from pymoo.operators.sampling.rnd import IntegerRandomSampling
    PYMOO_AVAILABLE = True
    print("pymoo disponible — NSGA-II et GA activés.")
except ImportError:
    PYMOO_AVAILABLE = False
    print("pymoo non installé. NSGA-II et GA simulés via Optuna NSGAIISampler.")

Librairies de base chargées.
pymoo disponible — NSGA-II et GA activés.


## 1. Chargement des données

In [2]:
X_train = np.load("data_prepared/X_train_optim.npy")   # 10 000 images
y_train = np.load("data_prepared/y_train_optim.npy")
X_val   = np.load("data_prepared/X_val_flat.npy")
y_val   = np.load("data_prepared/y_val_flat.npy")
X_test  = np.load("data_prepared/X_test_flat.npy")
y_test  = np.load("data_prepared/y_test_flat.npy")

print(f"Train (optim): {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}")

Train (optim): (10000, 784)  |  Val: (12000, 784)  |  Test: (10000, 784)


## 2. Fonction d'évaluation commune

In [3]:
# Mapping des hyperparamètres
N_ESTIMATORS_CHOICES = [10, 25, 50, 100, 150, 200, 300]
MAX_DEPTH_CHOICES    = [5, 8, 10, 15, 20, None]
MIN_SAMPLES_CHOICES  = [2, 5, 10, 20]
MAX_FEATURES_CHOICES = ["sqrt", "log2", 0.3, 0.5]

def evaluate_rf_params(n_est_idx, max_depth_idx, min_samples_idx, max_features_idx):
    """
    Entraîne un Random Forest avec les hyperparamètres spécifiés (indices).
    Retourne (accuracy_val, inference_time_ms_per_image).
    """
    n_est       = N_ESTIMATORS_CHOICES[int(n_est_idx)]
    max_depth   = MAX_DEPTH_CHOICES[int(max_depth_idx)]
    min_samples = MIN_SAMPLES_CHOICES[int(min_samples_idx)]
    max_feat    = MAX_FEATURES_CHOICES[int(max_features_idx)]

    model = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=max_depth,
        min_samples_split=min_samples,
        max_features=max_feat,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # Inférence sur val
    t0 = time.time()
    y_pred = model.predict(X_val)
    inf_ms = (time.time() - t0) / len(X_val) * 1000

    acc = accuracy_score(y_val, y_pred)

    params_dict = {
        "n_estimators": n_est,
        "max_depth": max_depth,
        "min_samples_split": min_samples,
        "max_features": max_feat
    }
    return acc, inf_ms, params_dict

print("Espace de recherche :")
print(f"  n_estimators     : {N_ESTIMATORS_CHOICES}")
print(f"  max_depth        : {MAX_DEPTH_CHOICES}")
print(f"  min_samples_split: {MIN_SAMPLES_CHOICES}")
print(f"  max_features     : {MAX_FEATURES_CHOICES}")
print(f"  Total combinaisons possibles : {len(N_ESTIMATORS_CHOICES)*len(MAX_DEPTH_CHOICES)*len(MIN_SAMPLES_CHOICES)*len(MAX_FEATURES_CHOICES)}")

Espace de recherche :
  n_estimators     : [10, 25, 50, 100, 150, 200, 300]
  max_depth        : [5, 8, 10, 15, 20, None]
  min_samples_split: [2, 5, 10, 20]
  max_features     : ['sqrt', 'log2', 0.3, 0.5]
  Total combinaisons possibles : 672


## 2b. Fonctions d'évaluation — SVM et k-NN

Définition des espaces de recherche et des fonctions d'évaluation pour SVM (C, kernel, gamma) et k-NN (k, weights, metric).

In [4]:
# ============================================================
# Fonctions d'évaluation pour SVM et k-NN
# ============================================================

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# --- Espace de recherche SVM ---
C_CHOICES      = [0.1, 0.5, 1.0, 5.0, 10.0]
KERNEL_CHOICES = ["rbf", "linear", "poly"]
GAMMA_CHOICES  = ["scale", "auto", 0.001, 0.01]

def evaluate_svm_params(c_idx, kernel_idx, gamma_idx):
    """
    Entraîne un SVM et retourne (accuracy_val, training_time_s, params).
    Objectifs : maximiser accuracy + minimiser temps d'entraînement.
    """
    C_val      = C_CHOICES[int(c_idx)]
    kernel_val = KERNEL_CHOICES[int(kernel_idx)]
    gamma_val  = GAMMA_CHOICES[int(gamma_idx)]

    # Utiliser kernel="linear" avec gamma fixe si nécessaire
    model = SVC(C=C_val, kernel=kernel_val, gamma=gamma_val if kernel_val != "linear" else "scale",
                random_state=42)

    t0 = time.time()
    model.fit(X_train, y_train)
    train_time_s = time.time() - t0

    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)

    params = {
        "C": C_val,
        "kernel": kernel_val,
        "gamma": str(gamma_val),
    }
    return acc, train_time_s, params


# --- Espace de recherche k-NN ---
K_CHOICES      = [3, 5, 7, 11, 15, 21]
WEIGHTS_CHOICES = ["uniform", "distance"]
METRIC_CHOICES  = ["euclidean", "manhattan", "minkowski"]

def evaluate_knn_params(k_idx, weights_idx, metric_idx):
    """
    Entraîne un k-NN et retourne (accuracy_val, inference_time_ms, params).
    Objectifs : maximiser accuracy + minimiser temps de prédiction.
    """
    k_val       = K_CHOICES[int(k_idx)]
    weights_val = WEIGHTS_CHOICES[int(weights_idx)]
    metric_val  = METRIC_CHOICES[int(metric_idx)]

    model = KNeighborsClassifier(n_neighbors=k_val, weights=weights_val,
                                  metric=metric_val, n_jobs=-1)
    model.fit(X_train, y_train)

    t0 = time.time()
    y_pred = model.predict(X_val)
    inf_ms = (time.time() - t0) / len(X_val) * 1000

    acc = accuracy_score(y_val, y_pred)
    params = {
        "k": k_val,
        "weights": weights_val,
        "metric": metric_val,
    }
    return acc, inf_ms, params

print("Fonctions d'évaluation SVM et k-NN prêtes.")
print(f"SVM  — espace de recherche: {len(C_CHOICES)} x {len(KERNEL_CHOICES)} x {len(GAMMA_CHOICES)} = {len(C_CHOICES)*len(KERNEL_CHOICES)*len(GAMMA_CHOICES)} combinaisons")
print(f"k-NN — espace de recherche: {len(K_CHOICES)} x {len(WEIGHTS_CHOICES)} x {len(METRIC_CHOICES)} = {len(K_CHOICES)*len(WEIGHTS_CHOICES)*len(METRIC_CHOICES)} combinaisons")


Fonctions d'évaluation SVM et k-NN prêtes.
SVM  — espace de recherche: 5 x 3 x 4 = 60 combinaisons
k-NN — espace de recherche: 6 x 2 x 3 = 36 combinaisons


## 2c. Fonction d'évaluation — Gradient Boosting

Espace de recherche : n_estimators, learning_rate, max_depth.

In [5]:
# ============================================================
# Fonction d'évaluation — Gradient Boosting
# ============================================================

# Espace de recherche Gradient Boosting
N_ESTIMATORS_GB = [50, 100, 150, 200]
LEARNING_RATE_GB = [0.01, 0.05, 0.1, 0.2]
MAX_DEPTH_GB     = [2, 3, 4, 5]

def evaluate_gb_params(n_est_idx, lr_idx, depth_idx):
    """
    Entraîne un Gradient Boosting et retourne (accuracy_val, inference_time_ms, params).
    Objectifs : maximiser accuracy + minimiser temps d'inférence.
    """
    n_est = N_ESTIMATORS_GB[int(n_est_idx)]
    lr    = LEARNING_RATE_GB[int(lr_idx)]
    depth = MAX_DEPTH_GB[int(depth_idx)]

    model = GradientBoostingClassifier(
        n_estimators=n_est,
        learning_rate=lr,
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    t0 = time.time()
    y_pred = model.predict(X_val)
    inf_ms = (time.time() - t0) / len(X_val) * 1000

    acc = accuracy_score(y_val, y_pred)
    params = {
        "n_estimators": n_est,
        "learning_rate": lr,
        "max_depth": depth,
    }
    return acc, inf_ms, params

print("Fonction d'évaluation Gradient Boosting prête.")
print(f"GB — espace: {len(N_ESTIMATORS_GB)} x {len(LEARNING_RATE_GB)} x {len(MAX_DEPTH_GB)} = {len(N_ESTIMATORS_GB)*len(LEARNING_RATE_GB)*len(MAX_DEPTH_GB)} combinaisons")


Fonction d'évaluation Gradient Boosting prête.
GB — espace: 4 x 4 x 4 = 64 combinaisons


## 3. Méthode 1 — Bayesian Optimization (Optuna TPE)

TPE (Tree-structured Parzen Estimator) modélise la distribution des bons vs mauvais hyperparamètres et propose des candidats en zone prometteuse.

In [6]:
N_TRIALS_BAYESIAN = 20
bayesian_results = []

def objective_bayesian(trial):
    n_est_idx       = trial.suggest_int("n_est_idx", 0, len(N_ESTIMATORS_CHOICES)-1)
    max_depth_idx   = trial.suggest_int("max_depth_idx", 0, len(MAX_DEPTH_CHOICES)-1)
    min_samples_idx = trial.suggest_int("min_samples_idx", 0, len(MIN_SAMPLES_CHOICES)-1)
    max_feat_idx    = trial.suggest_int("max_feat_idx", 0, len(MAX_FEATURES_CHOICES)-1)

    acc, inf_ms, params = evaluate_rf_params(n_est_idx, max_depth_idx, min_samples_idx, max_feat_idx)

    bayesian_results.append({
        "trial": trial.number,
        "method": "Bayesian",
        "accuracy": acc,
        "inference_ms": inf_ms,
        **params
    })
    return acc, inf_ms

print(f"Lancement Bayesian Optimization ({N_TRIALS_BAYESIAN} trials)...")
t_start = time.time()

study_bayesian = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_bayesian.optimize(objective_bayesian, n_trials=N_TRIALS_BAYESIAN, show_progress_bar=True)

bayesian_time = time.time() - t_start
print(f"\nBayesian Optimization terminé en {bayesian_time:.1f}s")
print(f"Trials Pareto : {len(study_bayesian.best_trials)}")

Lancement Bayesian Optimization (20 trials)...


  0%|          | 0/20 [00:00<?, ?it/s]


Bayesian Optimization terminé en 260.4s
Trials Pareto : 5


## 4. Méthode 2 — Hyperband (Optuna)

Hyperband alloue progressivement plus de ressources aux configurations prometteuses et élimine rapidement les mauvaises (Successive Halving).

In [7]:
N_TRIALS_HYPERBAND = 20
hyperband_results = []
hyperband_trial_counter = [0]

def objective_hyperband(trial):
    n_est_idx       = trial.suggest_int("n_est_idx", 0, len(N_ESTIMATORS_CHOICES)-1)
    max_depth_idx   = trial.suggest_int("max_depth_idx", 0, len(MAX_DEPTH_CHOICES)-1)
    min_samples_idx = trial.suggest_int("min_samples_idx", 0, len(MIN_SAMPLES_CHOICES)-1)
    max_feat_idx    = trial.suggest_int("max_feat_idx", 0, len(MAX_FEATURES_CHOICES)-1)

    acc, inf_ms, params = evaluate_rf_params(n_est_idx, max_depth_idx, min_samples_idx, max_feat_idx)

    hyperband_results.append({
        "trial": hyperband_trial_counter[0],
        "method": "Hyperband",
        "accuracy": acc,
        "inference_ms": inf_ms,
        **params
    })
    hyperband_trial_counter[0] += 1
    return acc, inf_ms

print(f"Lancement Hyperband ({N_TRIALS_HYPERBAND} trials)...")
t_start = time.time()

study_hyperband = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=42)  # Approche adaptative multi-objectifs
)
study_hyperband.optimize(objective_hyperband, n_trials=N_TRIALS_HYPERBAND, show_progress_bar=True)

hyperband_time = time.time() - t_start
print(f"\nHyperband terminé en {hyperband_time:.1f}s")
print(f"Trials Pareto : {len(study_hyperband.best_trials)}")

Lancement Hyperband (20 trials)...


  0%|          | 0/20 [00:00<?, ?it/s]


Hyperband terminé en 483.6s
Trials Pareto : 6


## 5. Méthode 3 — NSGA-II (via Optuna NSGAIISampler ou pymoo)

NSGA-II (Non-dominated Sorting Genetic Algorithm II) : algorithme évolutionnaire multi-objectifs qui maintient une population et utilise le tri non-dominé + crowding distance pour sélectionner les meilleures solutions.

In [8]:
N_TRIALS_NSGA2 = 20
nsga2_results = []
nsga2_trial_counter = [0]

def objective_nsga2(trial):
    n_est_idx       = trial.suggest_int("n_est_idx", 0, len(N_ESTIMATORS_CHOICES)-1)
    max_depth_idx   = trial.suggest_int("max_depth_idx", 0, len(MAX_DEPTH_CHOICES)-1)
    min_samples_idx = trial.suggest_int("min_samples_idx", 0, len(MIN_SAMPLES_CHOICES)-1)
    max_feat_idx    = trial.suggest_int("max_feat_idx", 0, len(MAX_FEATURES_CHOICES)-1)

    acc, inf_ms, params = evaluate_rf_params(n_est_idx, max_depth_idx, min_samples_idx, max_feat_idx)

    nsga2_results.append({
        "trial": nsga2_trial_counter[0],
        "method": "NSGA-II",
        "accuracy": acc,
        "inference_ms": inf_ms,
        **params
    })
    nsga2_trial_counter[0] += 1
    return acc, inf_ms

print(f"Lancement NSGA-II ({N_TRIALS_NSGA2} trials)...")
t_start = time.time()

study_nsga2 = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(
        population_size=10,
        mutation_prob=0.1,
        crossover_prob=0.9,
        seed=42
    )
)
study_nsga2.optimize(objective_nsga2, n_trials=N_TRIALS_NSGA2, show_progress_bar=True)

nsga2_time = time.time() - t_start
print(f"\nNSGA-II terminé en {nsga2_time:.1f}s")
print(f"Trials Pareto : {len(study_nsga2.best_trials)}")

Lancement NSGA-II (20 trials)...


  0%|          | 0/20 [00:00<?, ?it/s]


NSGA-II terminé en 434.6s
Trials Pareto : 4


## 6. Méthode 4 — Genetic Algorithm

Algorithme génétique : évolution d'une population de configurations par sélection, croisement et mutation. Optimisation mono-objectif sur un score combiné (accuracy - pénalité temps d'inférence).

In [9]:
import random

N_TRIALS_GA = 20
ga_results = []

class GeneticOptimizer:
    """Algorithme génétique simple pour l'optimisation des hyperparamètres."""

    def __init__(self, pop_size=8, n_generations=5, mutation_rate=0.2, seed=42):
        self.pop_size = pop_size
        self.n_generations = n_generations
        self.mutation_rate = mutation_rate
        self.bounds = [
            (0, len(N_ESTIMATORS_CHOICES)-1),
            (0, len(MAX_DEPTH_CHOICES)-1),
            (0, len(MIN_SAMPLES_CHOICES)-1),
            (0, len(MAX_FEATURES_CHOICES)-1)
        ]
        random.seed(seed)
        np.random.seed(seed)

    def _random_individual(self):
        return [random.randint(lo, hi) for lo, hi in self.bounds]

    def _fitness(self, individual):
        acc, inf_ms, params = evaluate_rf_params(*individual)
        # Score combiné : maximiser accuracy, pénaliser le temps d'inférence
        score = acc - 0.01 * inf_ms
        ga_results.append({
            "trial": len(ga_results),
            "method": "Genetic Algorithm",
            "accuracy": acc,
            "inference_ms": inf_ms,
            "fitness": score,
            **params
        })
        return score

    def _crossover(self, p1, p2):
        point = random.randint(1, len(p1)-1)
        return p1[:point] + p2[point:], p2[:point] + p1[point:]

    def _mutate(self, individual):
        ind = individual.copy()
        for i, (lo, hi) in enumerate(self.bounds):
            if random.random() < self.mutation_rate:
                ind[i] = random.randint(lo, hi)
        return ind

    def run(self):
        print(f"  Génération initiale ({self.pop_size} individus)...")
        population = [self._random_individual() for _ in range(self.pop_size)]
        fitness_scores = [self._fitness(ind) for ind in population]

        for gen in range(self.n_generations):
            # Sélection par tournoi
            new_pop = []
            for _ in range(self.pop_size // 2):
                idx1, idx2 = random.sample(range(self.pop_size), 2)
                p1 = population[idx1] if fitness_scores[idx1] > fitness_scores[idx2] else population[idx2]
                idx3, idx4 = random.sample(range(self.pop_size), 2)
                p2 = population[idx3] if fitness_scores[idx3] > fitness_scores[idx4] else population[idx4]

                c1, c2 = self._crossover(p1, p2)
                new_pop.extend([self._mutate(c1), self._mutate(c2)])

            new_fitness = [self._fitness(ind) for ind in new_pop]

            # Garder les meilleurs (élitisme)
            combined_pop = population + new_pop
            combined_fit = fitness_scores + new_fitness
            sorted_idx = np.argsort(combined_fit)[::-1][:self.pop_size]
            population = [combined_pop[i] for i in sorted_idx]
            fitness_scores = [combined_fit[i] for i in sorted_idx]

            print(f"  Génération {gen+1}/{self.n_generations} | Meilleur fitness: {fitness_scores[0]:.4f}")

        best_ind = population[0]
        return best_ind, fitness_scores[0]


print(f"Lancement Genetic Algorithm...")
t_start = time.time()

ga = GeneticOptimizer(pop_size=8, n_generations=4, mutation_rate=0.25, seed=42)
best_individual_ga, best_fitness_ga = ga.run()

ga_time = time.time() - t_start
print(f"\nGenetic Algorithm terminé en {ga_time:.1f}s")
best_ga_acc, best_ga_inf, best_ga_params = evaluate_rf_params(*best_individual_ga)
print(f"Meilleur individu : accuracy={best_ga_acc:.4f}, inférence={best_ga_inf:.5f} ms")
print(f"Hyperparamètres   : {best_ga_params}")

Lancement Genetic Algorithm...
  Génération initiale (8 individus)...
  Génération 1/4 | Meilleur fitness: 0.8566
  Génération 2/4 | Meilleur fitness: 0.8604
  Génération 3/4 | Meilleur fitness: 0.8604
  Génération 4/4 | Meilleur fitness: 0.8604

Genetic Algorithm terminé en 598.5s
Meilleur individu : accuracy=0.8608, inférence=0.04336 ms
Hyperparamètres   : {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'max_features': 'sqrt'}


## 3b. Optimisation Multi-Objectifs — SVM (Accuracy vs Temps d'entraînement)

L'Option 2 demande d'optimiser le SVM selon **accuracy et temps d'entraînement**.
On utilise Bayesian Optimization (TPE) avec Optuna en multi-objectif.

In [10]:
N_TRIALS_SVM = 20
svm_results = []

def objective_svm(trial):
    c_idx      = trial.suggest_int("c_idx",      0, len(C_CHOICES)-1)
    kernel_idx = trial.suggest_int("kernel_idx", 0, len(KERNEL_CHOICES)-1)
    gamma_idx  = trial.suggest_int("gamma_idx",  0, len(GAMMA_CHOICES)-1)

    acc, train_s, params = evaluate_svm_params(c_idx, kernel_idx, gamma_idx)

    svm_results.append({
        "trial": trial.number,
        "model": "SVM",
        "method": "Bayesian",
        "accuracy": acc,
        "train_time_s": train_s,
        **params
    })
    return acc, train_s   # maximize acc, minimize train_time

print(f"Lancement Bayesian Optimization SVM ({N_TRIALS_SVM} trials)...")
t_start_svm = time.time()

study_svm = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_svm.optimize(objective_svm, n_trials=N_TRIALS_SVM, show_progress_bar=False)
svm_time = time.time() - t_start_svm

svm_df = pd.DataFrame(svm_results)
print(f"SVM optimisation terminée en {svm_time:.1f}s")
print(svm_df[["accuracy", "train_time_s", "C", "kernel"]].describe().round(4))


Lancement Bayesian Optimization SVM (20 trials)...
SVM optimisation terminée en 2024.3s
       accuracy  train_time_s       C
count   20.0000       20.0000  20.000
mean     0.7639       41.8006   3.090
std      0.1852       38.1423   3.549
min      0.1138       18.4170   0.100
25%      0.7976       21.5044   0.500
50%      0.8185       25.0624   1.000
75%      0.8431       39.0250   5.000
max      0.8817      153.0021  10.000


## 3c. Optimisation Multi-Objectifs — k-NN (Accuracy vs Temps de prédiction)

L'Option 2 demande d'optimiser le k-NN selon **accuracy et temps de prédiction** (inférence).
On utilise Bayesian Optimization (TPE) multi-objectif.

In [11]:
N_TRIALS_KNN = 20
knn_results = []

def objective_knn(trial):
    k_idx       = trial.suggest_int("k_idx",       0, len(K_CHOICES)-1)
    weights_idx = trial.suggest_int("weights_idx", 0, len(WEIGHTS_CHOICES)-1)
    metric_idx  = trial.suggest_int("metric_idx",  0, len(METRIC_CHOICES)-1)

    acc, inf_ms, params = evaluate_knn_params(k_idx, weights_idx, metric_idx)

    knn_results.append({
        "trial": trial.number,
        "model": "k-NN",
        "method": "Bayesian",
        "accuracy": acc,
        "inference_ms": inf_ms,
        **params
    })
    return acc, inf_ms   # maximize acc, minimize inference_ms

print(f"Lancement Bayesian Optimization k-NN ({N_TRIALS_KNN} trials)...")
t_start_knn = time.time()

study_knn = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_knn.optimize(objective_knn, n_trials=N_TRIALS_KNN, show_progress_bar=False)
knn_time = time.time() - t_start_knn

knn_df = pd.DataFrame(knn_results)
print(f"k-NN optimisation terminée en {knn_time:.1f}s")
print(knn_df[["accuracy", "inference_ms", "k", "weights", "metric"]].describe().round(4))


Lancement Bayesian Optimization k-NN (20 trials)...
k-NN optimisation terminée en 407.3s
       accuracy  inference_ms        k
count   20.0000       20.0000  20.0000
mean     0.8290        1.6952   9.4000
std      0.0058        1.6499   5.4522
min      0.8179        0.3186   3.0000
25%      0.8268        0.3324   5.0000
50%      0.8292        0.3710   7.0000
75%      0.8321        3.5819  12.0000
max      0.8382        3.8827  21.0000


## 3d. Optimisation Multi-Objectifs — Gradient Boosting (Accuracy vs Temps d'inférence)

L'Option 2 liste Gradient Boosting comme modèle adapté.
On optimise ses hyperparamètres (n_estimators, learning_rate, max_depth) selon accuracy + temps de prédiction.

In [ ]:
N_TRIALS_GB = 20
gb_results_optim = []

def objective_gb(trial):
    n_est_idx = trial.suggest_int("n_est_idx", 0, len(N_ESTIMATORS_GB)-1)
    lr_idx    = trial.suggest_int("lr_idx",    0, len(LEARNING_RATE_GB)-1)
    depth_idx = trial.suggest_int("depth_idx", 0, len(MAX_DEPTH_GB)-1)

    acc, inf_ms, params = evaluate_gb_params(n_est_idx, lr_idx, depth_idx)

    gb_results_optim.append({
        "trial": trial.number,
        "model": "Gradient Boosting",
        "method": "Bayesian",
        "accuracy": acc,
        "inference_ms": inf_ms,
        **params
    })
    return acc, inf_ms

print(f"Lancement Bayesian Optimization Gradient Boosting ({N_TRIALS_GB} trials)...")
t_start_gb = time.time()

study_gb = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_gb.optimize(objective_gb, n_trials=N_TRIALS_GB, show_progress_bar=False)
gb_optim_time = time.time() - t_start_gb

gb_optim_df = pd.DataFrame(gb_results_optim)
print(f"Gradient Boosting optimisation terminée en {gb_optim_time:.1f}s")
print(gb_optim_df[["accuracy", "inference_ms", "n_estimators", "learning_rate", "max_depth"]].describe().round(4))


Lancement Bayesian Optimization Gradient Boosting (20 trials)...


## 7. Consolidation de tous les résultats

In [ ]:
all_results = pd.DataFrame(
    bayesian_results + hyperband_results + nsga2_results + ga_results
)

os.makedirs("results", exist_ok=True)
all_results.to_csv("results/optuna_trials_results.csv", index=False)

print(f"Total trials : {len(all_results)}")
print(all_results.groupby("method")[["accuracy", "inference_ms"]].describe().round(4))

## 8. Frontière de Pareto — Accuracy vs Temps d'inférence

In [ ]:
def is_pareto_front(accuracies, times):
    """Retourne un masque booléen des solutions non-dominées."""
    n = len(accuracies)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            # j domine i si j a une meilleure accuracy ET un meilleur temps
            if accuracies[j] >= accuracies[i] and times[j] <= times[i]:
                if accuracies[j] > accuracies[i] or times[j] < times[i]:
                    is_pareto[i] = False
                    break
    return is_pareto

# Couleurs par méthode
method_colors = {
    "Bayesian": "#2E86AB",
    "Hyperband": "#E84855",
    "NSGA-II": "#3BB273",
    "Genetic Algorithm": "#F4A261"
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Graphique 1 : Toutes les solutions ---
ax = axes[0]
for method, color in method_colors.items():
    subset = all_results[all_results["method"] == method]
    ax.scatter(subset["inference_ms"], subset["accuracy"],
               c=color, label=method, alpha=0.7, s=60, edgecolors="white", linewidths=0.5)

# Frontière de Pareto globale
pareto_mask = is_pareto_front(all_results["accuracy"].values, all_results["inference_ms"].values)
pareto_pts = all_results[pareto_mask].sort_values("inference_ms")
ax.plot(pareto_pts["inference_ms"], pareto_pts["accuracy"],
        "k--", linewidth=2, label="Frontière de Pareto", zorder=5)
ax.scatter(pareto_pts["inference_ms"], pareto_pts["accuracy"],
           c="black", s=100, zorder=6, marker="*")

ax.set_xlabel("Temps d'inférence (ms/image)", fontsize=11)
ax.set_ylabel("Accuracy Validation", fontsize=11)
ax.set_title("Toutes les solutions — Accuracy vs Temps d'inférence", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- Graphique 2 : Frontière de Pareto par méthode ---
ax2 = axes[1]
for method, color in method_colors.items():
    subset = all_results[all_results["method"] == method]
    mask = is_pareto_front(subset["accuracy"].values, subset["inference_ms"].values)
    pareto = subset[mask].sort_values("inference_ms")
    ax2.scatter(subset["inference_ms"], subset["accuracy"],
                c=color, alpha=0.3, s=40)
    ax2.plot(pareto["inference_ms"], pareto["accuracy"],
             color=color, linewidth=2, marker="o", markersize=7, label=method)

ax2.set_xlabel("Temps d'inférence (ms/image)", fontsize=11)
ax2.set_ylabel("Accuracy Validation", fontsize=11)
ax2.set_title("Frontières de Pareto par méthode d'optimisation", fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle("Optimisation Multi-Objectifs — Random Forest sur Fashion-MNIST",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/pareto_front.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\n{len(pareto_pts)} solutions non-dominées sur {len(all_results)} trials total")

## 9. Tableau des solutions non-dominées

In [ ]:
pareto_table = all_results[pareto_mask][["method", "accuracy", "inference_ms",
                                          "n_estimators", "max_depth",
                                          "min_samples_split", "max_features"]].copy()
pareto_table = pareto_table.sort_values("accuracy", ascending=False).reset_index(drop=True)

pareto_table.to_csv("results/pareto_solutions.csv", index=False)

print("=== Solutions non-dominées (Frontière de Pareto globale) ===")
print(pareto_table.to_string(index=True))

## 7b. Frontières de Pareto — SVM et k-NN

Visualisation des frontières de Pareto pour SVM (accuracy vs temps d'entraînement) et k-NN (accuracy vs temps d'inférence).

In [ ]:
def is_pareto_front_generic(obj1_max, obj2_min):
    """Calcule la frontière de Pareto pour 2 objectifs : maximiser obj1, minimiser obj2."""
    n = len(obj1_max)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if obj1_max[j] >= obj1_max[i] and obj2_min[j] <= obj2_min[i]:
                if obj1_max[j] > obj1_max[i] or obj2_min[j] < obj2_min[i]:
                    is_pareto[i] = False
                    break
    return is_pareto

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- SVM : Accuracy vs Temps entraînement ---
ax = axes[0]
svm_pareto = is_pareto_front_generic(svm_df["accuracy"].values, svm_df["train_time_s"].values)
ax.scatter(svm_df["train_time_s"], svm_df["accuracy"], c="#E84855", alpha=0.6, s=60, label="Tous les trials")
pareto_svm = svm_df[svm_pareto].sort_values("train_time_s")
ax.plot(pareto_svm["train_time_s"], pareto_svm["accuracy"], "k--", linewidth=2, label="Frontière de Pareto")
ax.scatter(pareto_svm["train_time_s"], pareto_svm["accuracy"], c="black", s=100, zorder=6, marker="*")
ax.set_xlabel("Temps d'entraînement (s)", fontsize=11)
ax.set_ylabel("Accuracy Validation", fontsize=11)
ax.set_title("SVM — Accuracy vs Temps d'entraînement", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# --- k-NN : Accuracy vs Temps d'inférence ---
ax = axes[1]
knn_pareto = is_pareto_front_generic(knn_df["accuracy"].values, knn_df["inference_ms"].values)
ax.scatter(knn_df["inference_ms"], knn_df["accuracy"], c="#3BB273", alpha=0.6, s=60, label="Tous les trials")
pareto_knn = knn_df[knn_pareto].sort_values("inference_ms")
ax.plot(pareto_knn["inference_ms"], pareto_knn["accuracy"], "k--", linewidth=2, label="Frontière de Pareto")
ax.scatter(pareto_knn["inference_ms"], pareto_knn["accuracy"], c="black", s=100, zorder=6, marker="*")
ax.set_xlabel("Temps d'inférence (ms/image)", fontsize=11)
ax.set_ylabel("Accuracy Validation", fontsize=11)
ax.set_title("k-NN — Accuracy vs Temps de prédiction", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("Frontières de Pareto — SVM et k-NN sur Fashion-MNIST", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/pareto_svm_knn.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"SVM  : {svm_pareto.sum()} solutions non-dominées sur {len(svm_df)} trials")
print(f"k-NN : {knn_pareto.sum()} solutions non-dominées sur {len(knn_df)} trials")


## 7c. Sauvegarde des résultats SVM et k-NN

In [ ]:
# Meilleur compromis SVM (score normalisé)
svm_acc_n = (svm_df["accuracy"] - svm_df["accuracy"].min()) / (svm_df["accuracy"].max() - svm_df["accuracy"].min() + 1e-9)
svm_time_n = (svm_df["train_time_s"] - svm_df["train_time_s"].min()) / (svm_df["train_time_s"].max() - svm_df["train_time_s"].min() + 1e-9)
svm_df["compromise_score"] = svm_acc_n - svm_time_n
best_svm = svm_df.loc[svm_df["compromise_score"].idxmax()]

# Meilleur compromis k-NN (score normalisé)
knn_acc_n = (knn_df["accuracy"] - knn_df["accuracy"].min()) / (knn_df["accuracy"].max() - knn_df["accuracy"].min() + 1e-9)
knn_inf_n = (knn_df["inference_ms"] - knn_df["inference_ms"].min()) / (knn_df["inference_ms"].max() - knn_df["inference_ms"].min() + 1e-9)
knn_df["compromise_score"] = knn_acc_n - knn_inf_n
best_knn = knn_df.loc[knn_df["compromise_score"].idxmax()]

# Sauvegardes CSV
svm_df.to_csv("results/svm_trials_results.csv", index=False)
knn_df.to_csv("results/knn_trials_results.csv", index=False)
svm_df[svm_pareto].to_csv("results/pareto_solutions_svm.csv", index=False)
knn_df[knn_pareto].to_csv("results/pareto_solutions_knn.csv", index=False)

print("=== Meilleur compromis SVM ===")
print(f"  Accuracy    : {best_svm['accuracy']:.4f}")
print(f"  Train time  : {best_svm['train_time_s']:.2f}s")
print(f"  C={best_svm['C']}, kernel={best_svm['kernel']}, gamma={best_svm['gamma']}")

print("\n=== Meilleur compromis k-NN ===")
print(f"  Accuracy      : {best_knn['accuracy']:.4f}")
print(f"  Inférence ms  : {best_knn['inference_ms']:.4f}")
print(f"  k={int(best_knn['k'])}, weights={best_knn['weights']}, metric={best_knn['metric']}")


## 7d. Frontière de Pareto — Gradient Boosting

In [ ]:
def is_pareto_front_generic2(obj1_max, obj2_min):
    n = len(obj1_max)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if obj1_max[j] >= obj1_max[i] and obj2_min[j] <= obj2_min[i]:
                if obj1_max[j] > obj1_max[i] or obj2_min[j] < obj2_min[i]:
                    is_pareto[i] = False
                    break
    return is_pareto

gb_pareto = is_pareto_front_generic2(gb_optim_df["accuracy"].values, gb_optim_df["inference_ms"].values)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(gb_optim_df["inference_ms"], gb_optim_df["accuracy"],
           c="#9B59B6", alpha=0.6, s=60, label="Tous les trials")
pareto_gb_pts = gb_optim_df[gb_pareto].sort_values("inference_ms")
ax.plot(pareto_gb_pts["inference_ms"], pareto_gb_pts["accuracy"],
        "k--", linewidth=2, label="Frontière de Pareto")
ax.scatter(pareto_gb_pts["inference_ms"], pareto_gb_pts["accuracy"],
           c="black", s=100, zorder=6, marker="*")
ax.set_xlabel("Temps d'inférence (ms/image)", fontsize=11)
ax.set_ylabel("Accuracy Validation", fontsize=11)
ax.set_title("Gradient Boosting — Accuracy vs Temps d'inférence", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/pareto_gb.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"GB : {gb_pareto.sum()} solutions non-dominées sur {len(gb_optim_df)} trials")


## 7e. Sauvegarde des résultats Gradient Boosting

In [ ]:
gb_acc_n  = (gb_optim_df["accuracy"] - gb_optim_df["accuracy"].min()) / (gb_optim_df["accuracy"].max() - gb_optim_df["accuracy"].min() + 1e-9)
gb_inf_n  = (gb_optim_df["inference_ms"] - gb_optim_df["inference_ms"].min()) / (gb_optim_df["inference_ms"].max() - gb_optim_df["inference_ms"].min() + 1e-9)
gb_optim_df["compromise_score"] = gb_acc_n - gb_inf_n
best_gb = gb_optim_df.loc[gb_optim_df["compromise_score"].idxmax()]

gb_optim_df.to_csv("results/gb_trials_results.csv", index=False)
gb_optim_df[gb_pareto].to_csv("results/pareto_solutions_gb.csv", index=False)

print("=== Meilleur compromis Gradient Boosting ===")
print(f"  Accuracy      : {best_gb['accuracy']:.4f}")
print(f"  Inférence ms  : {best_gb['inference_ms']:.4f}")
print(f"  n_estimators={int(best_gb['n_estimators'])}, lr={best_gb['learning_rate']}, max_depth={int(best_gb['max_depth'])}")


## 10. Comparaison des méthodes d'optimisation

In [ ]:
method_stats = []
search_times = {
    "Bayesian": bayesian_time,
    "Hyperband": hyperband_time,
    "NSGA-II": nsga2_time,
    "Genetic Algorithm": ga_time
}

for method in ["Bayesian", "Hyperband", "NSGA-II", "Genetic Algorithm"]:
    subset = all_results[all_results["method"] == method]
    pareto_m = is_pareto_front(subset["accuracy"].values, subset["inference_ms"].values)

    method_stats.append({
        "Méthode": method,
        "Nb trials": len(subset),
        "Best accuracy": subset["accuracy"].max(),
        "Min inférence ms": subset["inference_ms"].min(),
        "Nb solutions Pareto": pareto_m.sum(),
        "Temps recherche (s)": round(search_times[method], 1),
        "Temps moy/trial (s)": round(search_times[method] / len(subset), 2)
    })

method_df = pd.DataFrame(method_stats)
method_df.to_csv("results/method_comparison.csv", index=False)

print("=== Comparaison des méthodes d'optimisation ===")
print(method_df.to_string(index=False))

## 11. Sauvegarde du meilleur trial (compromis accuracy/temps)

In [ ]:
# Choix du meilleur compromis : score normalisé
pareto_df = all_results[pareto_mask].copy()
acc_norm = (pareto_df["accuracy"] - pareto_df["accuracy"].min()) / (pareto_df["accuracy"].max() - pareto_df["accuracy"].min() + 1e-9)
inf_norm = (pareto_df["inference_ms"] - pareto_df["inference_ms"].min()) / (pareto_df["inference_ms"].max() - pareto_df["inference_ms"].min() + 1e-9)
pareto_df["compromise_score"] = acc_norm - inf_norm

best_row = pareto_df.loc[pareto_df["compromise_score"].idxmax()]

best_info = pd.DataFrame([{
    "best_trial": int(best_row["trial"]) if "trial" in best_row else 0,
    "method": best_row["method"],
    "val_accuracy": round(best_row["accuracy"], 6),
    "inference_ms": round(best_row["inference_ms"], 6),
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": best_row["max_depth"],
    "min_samples_split": int(best_row["min_samples_split"]),
    "max_features": best_row["max_features"],
}])

best_info.to_csv("results/best_optuna_trial.csv", index=False)

print("=== Meilleur compromis sélectionné ===")
print(best_info.to_string(index=False))